In [ ]:
# !pip install gensim

import nltk
import numpy as np

# nltk.download('punkt_tab')

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,Dense

from gensim.models import Word2Vec

#Part 1 - Word2Vec

##Task 1

Train a Word2Vec model using vector_size=30, window=2, min_count=1

In [ ]:
texts = [
    "AI is changing the world",
    "Machine learning is amazing",
    "Deep learning powers many applications",
    "I enjoy learning NLP",
    "NLP is fun and useful"
]

In [ ]:
sentences = [text.lower().split() for text in texts]
print(sentences)

[['ai', 'is', 'changing', 'the', 'world'], ['machine', 'learning', 'is', 'amazing'], ['deep', 'learning', 'powers', 'many', 'applications'], ['i', 'enjoy', 'learning', 'nlp'], ['nlp', 'is', 'fun', 'and', 'useful']]


In [ ]:
model = Word2Vec(sentences, vector_size=30, window=2, min_count=1)

##Task 2

Find the number of unique words and identify the most frequent word without printing the entire vocabulary.

In [ ]:
# find unique words
num_unique_words = len(model.wv.key_to_index)
print("Number of unique words:", num_unique_words)

Number of unique words: 18


In [ ]:
# find the most frequent words
most_frequent_word = max(model.wv.key_to_index,key=lambda word: model.wv.get_vecattr(word, "count"))

count = model.wv.get_vecattr(most_frequent_word, "count")

print("Most frequent word:", most_frequent_word)

Most frequent word: learning


##Task 3

Print the word vectors of any two words of your choice.

In [ ]:
print(model.wv["ai"])

[-0.01666471 -0.00762281 -0.02416763 -0.03201106 -0.00914543 -0.02787613
 -0.02012959 -0.0189031  -0.00781379 -0.00568999 -0.02985666 -0.00245066
  0.02717502  0.02563477 -0.02402038 -0.01222277  0.01039517 -0.03190241
  0.00492146  0.02174822  0.01915473 -0.02921021 -0.01505715 -0.02713387
  0.00015319  0.03087878  0.01991102  0.01689103  0.01687021 -0.01080972]


In [ ]:
print(model.wv["world"])

[ 0.00257777 -0.02831926  0.02603269  0.03085764 -0.00914109  0.00266741
  0.00248884  0.01825962 -0.02868693  0.00194819  0.02289807  0.00743865
  0.00374892 -0.03107385  0.02827456 -0.02088043 -0.00997458  0.01164596
 -0.00257543  0.0047043   0.00593997 -0.022763   -0.03241604  0.03013528
  0.02066018 -0.02304309  0.01134494  0.00068688  0.01584582 -0.02373314]


##Task 4

Find the two most similar words to "learning".

In [ ]:
print(model.wv.most_similar("learning"))

[('amazing', 0.42548155784606934), ('applications', 0.18793466687202454), ('changing', 0.18696777522563934), ('world', 0.14677311480045319), ('many', 0.14138823747634888), ('i', 0.05848854407668114), ('powers', 0.03956747427582741), ('the', 0.004839003086090088), ('nlp', 0.003974487539380789), ('enjoy', -0.05643794313073158)]


most frequent words: amazing, applications

##Task 5

Choose any two pairs of words, calculate their similarity, and briefly explain the results.

In [ ]:
print(model.wv.similarity("nlp", "learning"))

0.0039744903


###Explain
This similarity is very close to 0, which means that "nlp" and "learning" are almost unrelated

In [ ]:
print(model.wv.similarity("fun", "enjoy"))

-0.41519612


###Explain
A negative similarity indicates that the model considers "fun" and "enjoy" to be dissimilar.

Even though these words have similar meanings in English, they never appear together in the same context in the dataset

##Observation
Changing the window doesn't change anything.

Changing the vector_size changed:
- most similar words to the word "learning"
- the similarity between "nlp","learning" & "fun","enjoy.

Increasing the min_count:
made the number of unqiue words drops from 18 to 3 to 2

#Part 2 - RNN

##Task 1

Convert the text into sequences using TensorFlow Tokenizer and print only the sequences.

In [ ]:
texts = [
    "I love AI",
    "AI is amazing",
    "I hate bugs",
    "Coding is fun",
    "This movie is terrible",
    "This movie is great"
]

labels = [1,1,0,1,0,1]

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

print(tokenizer.word_index)

{'is': 1, 'i': 2, 'ai': 3, 'this': 4, 'movie': 5, 'love': 6, 'amazing': 7, 'hate': 8, 'bugs': 9, 'coding': 10, 'fun': 11, 'terrible': 12, 'great': 13}


In [ ]:
sequences = tokenizer.texts_to_sequences(texts)

##Task 2

Pad all sequences to the maximum sentence length in the dataset.

In [ ]:
X = pad_sequences(sequences,maxlen=4)

print(X)

[[ 0  2  6  3]
 [ 0  3  1  7]
 [ 0  2  8  9]
 [ 0 10  1 11]
 [ 4  5  1 12]
 [ 4  5  1 13]]


##Task 3

Build a model with Embedding, SimpleRNN, and Dense output layers. Choose the embedding dimension and the number of RNN units yourself.

In [ ]:
model = Sequential()

model.add(
    Embedding(
        input_dim=20,
        output_dim=8,
        input_length=2
    )
)

model.add(SimpleRNN(16))

model.add(Dense(1,activation="sigmoid"))

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

##Task 4

Train the model for 15 epochs.

In [ ]:
model.fit(
    X,
    np.array(labels),
    epochs=15,
    verbose=1
)

Epoch 1/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.8333 - loss: 0.6836
Epoch 2/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8333 - loss: 0.6786
Epoch 3/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8333 - loss: 0.6737
Epoch 4/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.8333 - loss: 0.6688
Epoch 5/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8333 - loss: 0.6639
Epoch 6/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.8333 - loss: 0.6590
Epoch 7/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.8333 - loss: 0.6542
Epoch 8/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8333 - loss: 0.6494
Epoch 9/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8333 - loss: 0.6445
Epoch 10/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.6667 - loss: 0.6397
Epoch 11/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.6667 - loss: 0.6349
Epoch 12/15
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.6667 - loss: 0.6300
Epo

##Task 5

Predict the sentiment of any two new sentences of your choice and explain whether the predictions are reasonable.

In [ ]:
test = tokenizer.texts_to_sequences(
    ["AI Coding"]
)

test = pad_sequences(test,maxlen=4)

prediction = model.predict(test)

print(prediction)

if prediction > 0.5:
  print("Positive")
else:
  print("Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 450ms/step
[[0.5293837]]
Positive



"AI" and "Coding" both appears in a positive sentences so it's **reasonable** for the prediction to be positive.

In [ ]:
test = tokenizer.texts_to_sequences(
    ["amazing movie"]
)

test = pad_sequences(test,maxlen=4)

prediction = model.predict(test)

print(prediction)

if prediction > 0.5:
  print("Positive")
else:
  print("Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
[[0.56395954]]
Positive


"amazing" is in a positive sentence

"movie" is once in a positive sentence and once in negative sentence

so it's **reasonable** cause amazzing is in a positive sentence

#Bonus Challange

In [ ]:
texts = [
    "I love AI",
    "AI is amazing",
    "I hate bugs",
    "Coding is fun",
    "This movie is terrible",
    "This movie is great"
]

labels = [1,1,0,1,0,1]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(sequences,maxlen=4)

y = np.array(labels)

# different embedding dimensions
embedding_dims = [4, 6, 8, 10, 12]

print("-"*45)

for dim in embedding_dims:

    print(f"\nEmbedding Dimension = {dim}")

    model = Sequential()

    model.add(
        Embedding(
            input_dim=20,
            output_dim=dim,
            input_length=4
        )
    )

    model.add(SimpleRNN(16))
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(
        X,
        y,
        epochs=15,
        verbose=0
    )

    loss, accuracy = model.evaluate(X, y, verbose=0)

    print(f"Loss     : {loss:.4f}")
    print(f"Accuracy : {accuracy:.4f}")

---------------------------------------------

Embedding Dimension = 4
Loss     : 0.6212
Accuracy : 0.6667

Embedding Dimension = 6
Loss     : 0.6206
Accuracy : 0.6667

Embedding Dimension = 8
Loss     : 0.6259
Accuracy : 0.6667

Embedding Dimension = 10
Loss     : 0.6074
Accuracy : 0.6667

Embedding Dimension = 12
Loss     : 0.6341
Accuracy : 0.6667


####when changing output_dim
Accuracies are around: 0.167- 0.33 - 0.50 - 0.66 - 0.83.

Loss decrease from 0.7 to 0.6 with any outut dim.

All models converged to approximately 66.7% training accuracy after 15 epochs.

